# 03 · Structured extraction
**Objective:** extract company facts into a typed schema, and measure whether the model *invents* facts that the text
does not state. Notes contain a distractor company to tempt it. Data is fictional.

In [1]:
# Setup: works locally and on Google Colab. Never put secrets in notebook cells.
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = ""  # Set to your clone URL. On Colab, add HF_TOKEN under Secrets to use hosted models.
    if not Path("salesiq").exists():
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL in this cell to clone the SalesIQ repository.")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "salesiq"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml", "pydantic", "mcp"], check=True)
    ROOT = Path("salesiq").resolve()
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError:
        print("No HF_TOKEN secret; only a local model server can be used.")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())
sys.path.insert(0, str(ROOT / "apps" / "api"))
os.environ["SALESIQ_RESEARCH_DIR"] = str(ROOT / "research")

from app.agents.llm import LLMError, ModelRegistry

registry = ModelRegistry()
status = {s.key: s for s in registry.status()}
CANDIDATES = ["qwen3-4b", "qwen2.5-7b", "mistral-7b", "hf-qwen3-8b"]
MODEL = next((k for k in CANDIDATES if status[k].available), None)
if MODEL is None:
    # There is deliberately no mock: results must come from a real model.
    raise RuntimeError("No model is reachable. Start Ollama and pull a model, or set HF_TOKEN.\n"
                       + "\n".join(f"{s.key}: {s.reason}" for s in status.values() if not s.available))
print(f"Using real model: {MODEL} ({registry.model(MODEL).model})")

Using real model: qwen3-4b (hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M)


In [2]:
from app.agents.evaluation import baseline, metrics, tasks
from app.agents.evaluation.runner import Dataset

records = Dataset.load("train").prospects[:8]
gold = [r["gold"]["facts"] for r in records]
pred = []
for r in records:
    try:
        out, _ = registry.structured(MODEL, tasks.extraction_messages(r), tasks.ExtractionOut, max_tokens=200)
        pred.append(out.model_dump())
    except LLMError as exc:
        print(r["id"], exc.category)
        pred.append(None)
for name, p in (("regex", [baseline.extract(r).model_dump() for r in records]), (MODEL, pred)):
    rep = metrics.extraction_report(gold, p)
    print(f"{name:12} field accuracy {rep['field_accuracy']:.0%}, invented {rep['invented_fact_rate']:.0%} of {rep['absent_facts']} absent facts")
    print("   ", {k: f"{v:.0%}" for k, v in rep["per_field"].items()})

regex        field accuracy 100%, invented 0% of 2 absent facts
    {'company_name': '100%', 'employee_count': '100%', 'hq_city': '100%', 'industry': '100%', 'funding_stage': '100%'}
qwen3-4b     field accuracy 100%, invented 0% of 2 absent facts
    {'company_name': '100%', 'employee_count': '100%', 'hq_city': '100%', 'industry': '100%', 'funding_stage': '100%'}


## Output validation
Pydantic rejects malformed output, and a schema error is reported rather than repaired by guessing.

In [3]:
from pydantic import ValidationError
try:
    tasks.ExtractionOut.model_validate_json('{"company_name": "X", "employee_count": "many"}')
except ValidationError as exc:
    print("Rejected:", exc.error_count(), "problems")
print(pred[0]); print("gold:", gold[0])

Rejected: 4 problems
{'company_name': 'Wexley Cloud T17', 'employee_count': 5080, 'hq_city': 'Calgary', 'industry': 'B2B SaaS', 'funding_stage': None}
gold: {'company_name': 'Wexley Cloud T17', 'employee_count': 5080, 'hq_city': 'Calgary', 'industry': 'B2B SaaS', 'funding_stage': None}


**Expected:** high accuracy on this split, with an invented-fact rate reported separately because a confident wrong
value is worse than a null.

**Limitations:** templated fictional notes are far cleaner than real web pages. Real-document evaluation comes with Phase 5.